# Lecture 2 — Temperature: from weather to climate

## Working with real ERA5 monthly temperature data

In this notebook we move from the simple weather/climate examples of Lecture 1 to a real gridded climate dataset.

We will use **ERA5 monthly mean 2-m air temperature** (`t2m`). ERA5 provides a consistent, global reanalysis record extending back to 1940; the dataset used here contains monthly values from **1940–2025** over a regional European domain. 

The goal is not just to make a temperature plot. We will learn how a gridded climate dataset can be reduced to a **regional climate time series**, and then use that time series to study:

- the seasonal cycle
- climatology
- anomalies
- interannual variability
- the distribution of annual temperatures
- long-term trends
- a smoothed climate signal

> **Important:** this is monthly data. We are not reconstructing daily or hourly temperatures from it, so we will not calculate daily extremes such as hot days or frost days.

## 1. Load the data

The NetCDF file is placed in the repository's `data/` directory.

```text
climate-course/
├── environment.yml
├── notebooks/
│   └── 02_temperature.ipynb
└── data/
    └── ERA5_t2m_monthly.nc
```

The file used to develop this notebook contains the variable `t2m`, in Kelvin, with dimensions:

- `valid_time`
- `latitude`
- `longitude`

It covers **January 1940 to December 2025**.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import linregress

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12

# Find NetCDF files in the data directory
data_dir = Path("../data")
nc_files = sorted(data_dir.glob("ERA5_t2m_monthly.nc"))

if not nc_files:
    raise FileNotFoundError(
        "No NetCDF file found in ../data/. Put the ERA5 temperature file there first."
    )

nc_files

In [ ]:
# Open the first NetCDF file
ds = xr.open_dataset(nc_files[0])

ds

## 2. Inspect the dataset

Before analysing climate data, always inspect the dimensions, coordinates, units and variable names.

This is an important scientific habit: **do not assume the structure of a dataset before looking at it.**

In [ ]:
print("Dimensions:")
print(ds.dims)

print("\nCoordinates:")
print(list(ds.coords))

print("\nVariables:")
print(list(ds.data_vars))

print("\nTemperature variable:")
print(ds["t2m"])

print("\nTemperature units:", ds["t2m"].attrs.get("units"))

## 3. Convert temperature from Kelvin to °C

ERA5 `t2m` is provided in Kelvin.

For this course we will work in degrees Celsius:

\[
T_{°C} = T_K - 273.15
\]

We create a new variable so that the original data remain untouched.

In [ ]:
ds["temperature"] = ds["t2m"] - 273.15
ds["temperature"].attrs["units"] = "°C"

ds["temperature"].attrs

## 4. Look at the spatial temperature pattern

Let's first look at one month.

A climate dataset is not just a time series: it contains a **spatial field**.

The map below shows January 2025 temperature over the domain.

In [ ]:
# Select one month
temperature_jan2025 = ds["temperature"].sel(valid_time="2025-01")

temperature_jan2025.plot(
    cmap="coolwarm",
    cbar_kwargs={"label": "Temperature (°C)"}
)

plt.title("ERA5 temperature — January 2025")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

### Think about it

Why is temperature not spatially uniform?

Consider:

- latitude
- elevation
- land versus ocean
- proximity to the sea
- atmospheric circulation
- transport of heat by the atmosphere and ocean

## 5. From a map to a regional climate time series

A common climate-analysis problem is:

> **How do we turn a gridded temperature field into one temperature time series representing a region?**

We will calculate a **latitude-weighted regional mean**.

For a latitude-longitude grid, grid cells at different latitudes have different areas. A simple arithmetic mean would therefore give too much weight to high-latitude cells.

A common approximation is:

\[
w = \cos(\phi)
\]

where \(\phi\) is latitude.

We will use these weights when averaging over the selected region.

## 6. Choose a region

The dataset covers approximately 30–45°N and 10°W–40°E.

Try changing the coordinates below. This is one of the main exercises of the notebook.

For example:

- **Italy:** latitude 36–47°N, longitude 6–19°E
- **Northern Italy:** latitude 44–47°N, longitude 6–14°E
- **Central Mediterranean:** latitude 35–45°N, longitude 5–25°E
- **Eastern Mediterranean:** latitude 30–40°N, longitude 20–40°E

Remember that latitude may be stored in descending order, so we use `slice(max, min)` when necessary.

In [ ]:
# Choose your region
lat_min, lat_max = 36, 47
lon_min, lon_max = 6, 19

# The dataset latitude coordinate is descending
region = ds["temperature"].sel(
    latitude=slice(lat_max, lat_min),
    longitude=slice(lon_min, lon_max)
)

region

In [ ]:
# Check the selected region
region.isel(valid_time=0).plot(
    cmap="coolwarm",
    cbar_kwargs={"label": "Temperature (°C)"}
)

plt.title(f"Selected region: {lat_min}–{lat_max}°N, {lon_min}–{lon_max}°E")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## 7. Calculate the area-weighted regional mean

Now we reduce the 2-D spatial field to one value for every month.

In [ ]:
weights = np.cos(np.deg2rad(region.latitude))

regional_temperature = region.weighted(weights).mean(
    dim=("latitude", "longitude")
)

regional_temperature.name = "regional_temperature"

regional_temperature

## 8. Plot the regional temperature time series

This is now a **monthly climate time series**.

Notice the strong seasonal cycle. Individual months also vary from year to year.

In [ ]:
regional_temperature.plot()

plt.title("Regional monthly mean temperature")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 9. The seasonal cycle

A climate variable can have large predictable variations within a year.

Let's calculate the mean temperature for each calendar month over the full record.

In [ ]:
monthly_climatology = regional_temperature.groupby(
    "valid_time.month"
).mean()

monthly_climatology

In [ ]:
months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.plot(months, monthly_climatology.values, marker="o")

plt.title("Regional seasonal cycle")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

### Climate interpretation

The seasonal cycle tells us about the **typical annual rhythm** of temperature.

It is different from the weather of any particular year.

For example, July is usually warmer than January, but an individual July can still be unusually cool.

## 10. A 30-year climatology

Climate statistics are often calculated over a standard reference period.

Here we use **1991–2020**, a commonly used 30-year climate reference period.

We first calculate the mean temperature for each calendar month during this period.

In [ ]:
reference = regional_temperature.sel(
    valid_time=slice("1991-01-01", "2020-12-31")
)

climatology_1991_2020 = reference.groupby(
    "valid_time.month"
).mean()

plt.plot(months, climatology_1991_2020.values, marker="o")

plt.title("Regional temperature climatology, 1991–2020")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 11. Temperature anomalies

Absolute temperature depends strongly on location and season.

Anomalies are often easier to interpret when studying climate change.

For each month:

\[
T' = T - \overline{T}_{climatology}
\]

Here the climatological mean is the 1991–2020 average for that particular calendar month.

In [ ]:
climatology_for_month = climatology_1991_2020.sel(
    month=regional_temperature.valid_time.dt.month
)

anomalies = regional_temperature - climatology_for_month

anomalies.name = "temperature_anomaly"

anomalies.plot()

plt.axhline(0, linewidth=1)
plt.title("Regional monthly temperature anomalies")
plt.xlabel("Year")
plt.ylabel("Temperature anomaly (°C)")
plt.grid(alpha=0.25)
plt.show()

## 12. Annual mean temperature

The seasonal cycle can make long-term changes difficult to see.

We can average the monthly values within each year to obtain one annual mean temperature.

In [ ]:
annual_temperature = regional_temperature.groupby(
    "valid_time.year"
).mean()

annual_temperature

In [ ]:
plt.plot(
    annual_temperature.year,
    annual_temperature,
    marker="o",
    markersize=3,
    linewidth=1
)

plt.title("Annual mean regional temperature")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 13. Interannual variability

The annual mean temperature does not increase smoothly.

Individual years can be warmer or colder than the long-term trend.

This short-term variation is part of **climate variability**.

Let's look at the distribution of annual mean temperatures.

In [ ]:
plt.hist(
    annual_temperature.values,
    bins=15,
    edgecolor="black"
)

plt.xlabel("Annual mean temperature (°C)")
plt.ylabel("Number of years")
plt.title("Distribution of annual mean regional temperature")
plt.show()

## 14. Warmest and coldest years

Climate change is not only about the average trend. The distribution of temperatures also matters.

Let's identify the five warmest and five coldest years in the record.

In [ ]:
annual_df = annual_temperature.to_dataframe(
    name="temperature"
)

warmest = annual_df.sort_values("temperature", ascending=False).head(5)
coldest = annual_df.sort_values("temperature").head(5)

print("Warmest years:")
display(warmest)

print("\nColdest years:")
display(coldest)

## 15. Estimate the long-term trend

A simple first estimate of climate change is a linear regression:

\[
T(t) = a + bt
\]

where the slope \(b\) describes the change in temperature per year.

We will report the trend in **°C per decade**.

In [ ]:
years = annual_temperature.year.values
values = annual_temperature.values

slope, intercept, r_value, p_value, std_err = linregress(years, values)

trend_per_decade = slope * 10

print(f"Linear trend: {trend_per_decade:.2f} °C per decade")
print(f"R² = {r_value**2:.2f}")
print(f"p-value = {p_value:.3g}")

In [ ]:
trend_line = intercept + slope * years

plt.plot(years, values, label="Annual mean")
plt.plot(years, trend_line, linewidth=2, label="Linear trend")

plt.title("Long-term temperature trend")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Important interpretation

The regression line is **not a prediction of next year's temperature**.

It is a statistical description of the long-term change over the selected period.

The year-to-year deviations around the line represent variability that is not captured by a simple linear trend.

## 16. Separate the long-term signal from short-term variability

One way to make the low-frequency signal easier to see is to calculate a running mean.

Here we use an **11-year running mean**.

In [ ]:
running_mean = annual_temperature.rolling(
    year=11,
    center=True
).mean()

plt.plot(years, values, alpha=0.4, label="Annual mean")
plt.plot(
    running_mean.year,
    running_mean,
    linewidth=2,
    label="11-year running mean"
)

plt.title("Annual temperature and low-frequency variability")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

# Exercises

### Exercise 1 — Change the region

Choose a different region within the available domain.

Calculate its:

1. seasonal cycle
2. 1991–2020 climatology
3. annual mean temperature
4. linear trend

How does the trend compare with the original region?

---

### Exercise 2 — Change the reference period

Calculate anomalies relative to **1961–1990** instead of 1991–2020.

How does changing the baseline affect the anomaly values?

Does it change the underlying temperature trend?

---

### Exercise 3 — Warmest decades

Calculate the mean annual temperature for:

- 1951–1960
- 1961–1970
- 1971–1980
- ...
- 2011–2020

Plot the resulting decadal time series.

What do you notice?

---

### Exercise 4 — Think like a climate scientist

The regional annual temperature increases over time, but individual years can still be unusually cold.

Explain why **climate change does not mean that every year must be warmer than the previous year**.

Use the distinction between:

- climate trend
- interannual variability
- seasonal cycle
- anomalies

## Key takeaways

By the end of this notebook, we have moved through the full chain:

**gridded temperature → regional mean → seasonal cycle → climatology → anomaly → annual mean → variability → trend**

The important conceptual distinction is:

> **Weather describes individual events and short-term variations; climate describes the statistics and long-term behaviour of those variations.**

Real climate datasets allow us to quantify both.